# Experimento 1 — Lectura del dataset y preprocesamiento (TrashNet)

Alcance de `main`: solo lectura + prepro. Entrenamiento y app viven en `dev`.

Basado en [garythung/trashnet](https://github.com/garythung/trashnet) (Stanford CS 229). Dataset espejo: `kuchidareo/small_trashnet` — 2527 fotos, 6 clases (`glass, paper, cardboard, plastic, metal, trash`).

La lógica reusable está en `src/dataset.py` y `src/preprocessing.py`; aquí se muestra paso a paso.

## Paso 1 — Preparar librerías

In [ ]:
# Si falta alguna librería, descomenta la siguiente línea y ejecútala una vez:
# %pip install -r ../requirements.txt

import random
import numpy as np
import matplotlib.pyplot as plt
from torch.utils.data import DataLoader

from src.dataset import load_trashnet, contar_por_clase, split_indices, TrashDataset, SEED
from src.preprocessing import get_transforms, IMG_SIZE, BATCH

random.seed(SEED); np.random.seed(SEED)
print("IMG_SIZE:", IMG_SIZE, "| BATCH:", BATCH)

## Paso 2 — Descargar el dataset TrashNet

Descargamos gratis las 2527 fotos desde HuggingFace (espejo del dataset original del repo) y vemos cuántas hay por clase.

In [ ]:
hf, CLASES = load_trashnet()
print("Total imágenes:", len(hf))
print("Clases:", CLASES)

# Conteo por clase + gráfico
conteo = contar_por_clase(hf, CLASES)
print(conteo)

plt.figure(figsize=(8, 3.5))
plt.bar(conteo.keys(), conteo.values())
plt.title("Imágenes por clase (TrashNet: 2527 en total)")
plt.xticks(rotation=15)
plt.tight_layout()
plt.show()

## Paso 3 — Ver ejemplos

Miremos 12 fotos para conocer los datos.

In [ ]:
fig, axes = plt.subplots(3, 4, figsize=(10, 7))
for k, ax in enumerate(axes.flat):
    i = random.randint(0, len(hf) - 1)
    img = hf[i]["image"].convert("RGB")
    ax.imshow(img)
    ax.set_title(CLASES[int(hf[i]["label"])])
    ax.axis("off")
plt.suptitle("Ejemplos del dataset TrashNet")
plt.tight_layout()
plt.show()

## Paso 4 — Grupos train / val / test

Como en el paper original: **70% entrenar, 13% validar, 17% probar**. A las de entrenamiento les aplicamos aumentos (recorte, giro) para que la red aprenda mejor.

In [ ]:
train_tf, eval_tf = get_transforms(IMG_SIZE)

# División 70/13/17 con semilla fija (reproducible)
train_idx, val_idx, test_idx = split_indices(len(hf), seed=SEED)

train_ds = TrashDataset(hf, train_idx, train_tf)
val_ds = TrashDataset(hf, val_idx, eval_tf)
test_ds = TrashDataset(hf, test_idx, eval_tf)
train_ld = DataLoader(train_ds, batch_size=BATCH, shuffle=True)
val_ld = DataLoader(val_ds, batch_size=BATCH)
test_ld = DataLoader(test_ds, batch_size=BATCH)
print(f"train={len(train_ds)} val={len(val_ds)} test={len(test_ds)}")

# Ver un lote de entrenamiento
x, y = next(iter(train_ld))
print("Lote:", tuple(x.shape))
fig, axes = plt.subplots(2, 4, figsize=(10, 5))
media = np.array([0.485, 0.456, 0.406]); desv = np.array([0.229, 0.224, 0.225])
for k, ax in enumerate(axes.flat):
    foto = (x[k].numpy().transpose(1, 2, 0) * desv + media).clip(0, 1)
    ax.imshow(foto)
    ax.set_title(CLASES[int(y[k])])
    ax.axis("off")
plt.suptitle("Un lote de entrenamiento (con aumento de datos)")
plt.tight_layout()
plt.show()